In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import warnings
import torch

warnings.filterwarnings("ignore")
np.random.RandomState(42)
df = pd.read_csv("data/track_k_ecommerce_returns_train.csv")

df_train = pd.read_csv("data/track_k_ecommerce_returns_train.csv")
df_test = pd.read_csv("data/track_k_ecommerce_returns_test.csv")

X_train = df_train.drop(columns=['return_rate_pct'])
y_train = df_train[['return_rate_pct']]

X_test = df_test.drop(columns=['return_rate_pct'])
y_test = df_test[['return_rate_pct']]


In [4]:
# quick PyTorch NN train/eval (uses X_train, y_train, X_test, y_test from previous cell)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# prepare numeric data
X_tr = X_train.select_dtypes(include=[np.number]).fillna(0).values.astype(np.float32)
X_te = X_test.select_dtypes(include=[np.number]).fillna(0).values.astype(np.float32)
y_tr = y_train.values.astype(np.float32).reshape(-1, 1)
y_te = y_test.values.astype(np.float32).reshape(-1, 1)

# simple standardization
x_mean, x_std = X_tr.mean(axis=0, keepdims=True), X_tr.std(axis=0, keepdims=True) + 1e-8
X_tr = (X_tr - x_mean) / x_std
X_te = (X_te - x_mean) / x_std

# tensors and dataloaders
batch_size = 64
train_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
X_te_t = torch.from_numpy(X_te).to(device)
y_te_t = torch.from_numpy(y_te).to(device)

# model
in_dim = X_tr.shape[1]
model = torch.nn.Sequential(
    torch.nn.Linear(in_dim, 64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 1)
).to(device)

loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# train
epochs = 1000
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()

# evaluate MSE
model.eval()
with torch.no_grad():
    train_pred = model(torch.from_numpy(X_tr).to(device))
    test_pred = model(X_te_t)
    train_mse = loss_fn(train_pred, torch.from_numpy(y_tr).to(device)).item()
    test_mse = loss_fn(test_pred, y_te_t).item()

print(f"Train MSE: {train_mse:.6f} | Test MSE: {test_mse:.6f}")

Train MSE: 6.450144 | Test MSE: 38.612835
